# BGMI Tournament Winning Probability Prediction System

This notebook reviews the supplied BGIS/BMPS scorecards and the saved BMSD match snapshot, builds pre-match features using only earlier matches, compares calibrated probability models, evaluates a chronological holdout, and saves the artifacts used by Flask. Refresh BMSD scorecards on the Flask live page before rerunning the analysis to include newly published rounds.

The target is one-match WWCD probability, not championship probability. BMSD refreshes provide completed scorecards, stages, groups, finalists, and current player eliminations; future match forecasts appear only when the source publishes participating teams. BGIS winner labels are provided as a source win flag, without the original WWCD field.

In [ ]:
# 2. Import libraries
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd()
while not (ROOT / "src" / "data_preprocessing.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
from src.data_preprocessing import load_all_historical_data, load_bgis_data, load_bmps_data, load_bmsd_data, validate_match_lobbies
from src.feature_engineering import FEATURE_COLS, build_pre_match_features
from src.model_training import train_and_evaluate, RESULTS_DIR, MODEL_DIR
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print(ROOT)

In [ ]:
# 3. Load all actual tournament records
bgis = load_bgis_data(ROOT / "data")
bmps = load_bmps_data(ROOT / "data")
bmsd = load_bmsd_data(ROOT / "data")
history = load_all_historical_data(ROOT / "data")
print("BGIS, BMPS, BMSD:", bgis.shape, bmps.shape, bmsd.shape)
print("Combined:", history.shape)
display(history.head())

In [ ]:
# 4. Dataset overview
print("Shape:", history.shape, "Columns:", list(history.columns))
display(pd.DataFrame({"dtype": history.dtypes.astype(str), "missing": history.isna().sum()}))
display(history.groupby("tournament").agg(rows=("team", "size"), matches=("match_key", "nunique"), teams=("team_key", "nunique"), wins=("win", "sum")))
print("Duplicate rows:", history.duplicated().sum())
display(history[["points", "kills", "rank", "placement_points", "win"]].describe().T)

In [ ]:
# 5. Data cleaning and validation
# The shared loader converts types, standardizes fields, trims names, excludes incomplete BGIS
# fragments, and validates each lobby. Unsupported player-level fields are not imputed or invented.
validate_match_lobbies(history)
assert not history.duplicated(["match_key", "team_key"]).any()
assert history.groupby("match_key").win.sum().eq(1).all()
assert (history[["points", "kills"]].dropna() >= 0).all().all()
print("Validated matches:", history.match_key.nunique())

In [ ]:
# 6. Exploratory data analysis
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
history.groupby("tournament").match_key.nunique().plot.bar(ax=ax[0, 0], color="#7c5cff", title="Matches by tournament")
sns.histplot(data=history, x="kills", bins=15, ax=ax[0, 1]).set_title("Kills distribution")
sns.histplot(data=history, x="rank", discrete=True, ax=ax[1, 0]).set_title("Placement distribution")
sns.histplot(data=history, x="points", bins=20, ax=ax[1, 1]).set_title("Points distribution")
plt.tight_layout()
plt.show()
sns.heatmap(history[["points", "kills", "rank", "placement_points", "win"]].corr(), annot=True, cmap="mako")
plt.title("Scorecard feature correlations"); plt.tight_layout(); plt.show()
teams = history.groupby(["tournament", "team"], as_index=False).agg(matches=("match_key", "nunique"), avg_points=("points", "mean"), avg_kills=("kills", "mean"), wins=("win", "sum"))
display(teams.sort_values(["tournament", "avg_points"], ascending=[True, False]).groupby("tournament").head(8))
trend = history.groupby(["tournament", "match_order"], as_index=False).points.mean()
sns.lineplot(data=trend, x="match_order", y="points", hue="tournament", marker="o")
plt.title("Average points by match order"); plt.tight_layout(); plt.show()

In [ ]:
# 7. Leakage-safe feature engineering
features = build_pre_match_features(history)
display(features[["tournament", "match_id", "team", "win", *FEATURE_COLS]].head(20))
assert np.isfinite(features[FEATURE_COLS].to_numpy(float)).all()
print("Features use earlier completed matches only; same-match results do not enter their own rows.")

In [ ]:
# 8. Target creation and class imbalance
y = features["win"].astype(int)
display(y.value_counts().rename(index={0: "No WWCD", 1: "WWCD"}).to_frame("team-match rows"))
print("WWCD prevalence:", round(y.mean(), 4))
print("One winner per complete lobby; no resampling is applied so probability base rates stay natural.")

In [ ]:
# 9. Chronological train/test split
keys = features["match_key"].astype(str).reset_index(drop=True)
ordered = keys.drop_duplicates().tolist()
test_keys = set(ordered[-6:])
test_mask = keys.isin(test_keys)
print("Development matches:", len(ordered) - 6, "Holdout matches:", len(test_keys))
print("Holdout:", sorted(test_keys))

In [ ]:
# 10. Compare calibrated candidates and save a final model
result = train_and_evaluate(data_dir=ROOT / "data", model_dir=MODEL_DIR, results_dir=RESULTS_DIR)
print("Training completed.")

In [ ]:
# 11. Model comparison
comparison = result["comparison"]
display(comparison)
print("Selected by lowest chronological conditional match log loss, then Brier score.")

In [ ]:
# 12. Evaluation on later complete matches
display(pd.read_csv(RESULTS_DIR / "model_comparison.csv"))
print("Confusion matrix: rows are actual no-win/win, columns are predicted no-win/win")
display(pd.read_csv(RESULTS_DIR / "confusion_matrix.csv", index_col=0))
display(result["test_predictions"].head(16))

In [ ]:
# 13. Feature importance (model explanation, not causation)
importance = result["feature_importance"]
display(importance)
if not importance.empty:
    imp = importance.sort_values("importance").tail(10)
    plt.barh(imp.feature, imp.importance, color="#54b8ff")
    plt.title("Global feature importance"); plt.tight_layout(); plt.show()

In [ ]:
# 14. Probability calibration
chosen = comparison.loc[comparison.model.eq(result["selected_model"])].iloc[0]
print("Brier score:", chosen["brier_score"], "Log loss:", chosen["log_loss"])
print("Calibration uses sigmoid calibration on expanding chronological match-group folds.")
print((RESULTS_DIR / "model_metadata.json").read_text(encoding="utf-8"))
calibration_plot = RESULTS_DIR / "calibration_curve.png"
if calibration_plot.exists():
    plt.figure(figsize=(6, 5))
    plt.imshow(plt.imread(str(calibration_plot)))
    plt.axis("off")
    plt.show()

In [ ]:
# 15. Final model selection
print("Chosen model:", result["selected_model"])
display(comparison.set_index("model").loc[result["selected_model"]])
print("Small history and six-match holdout make these early estimates uncertain; they are not an accuracy guarantee.")

In [ ]:
# 16. Save and verify model artifacts
for path in (MODEL_DIR / "bgmi_winning_probability_model.joblib",
             MODEL_DIR / "preprocessing_pipeline.joblib",
             MODEL_DIR / "feature_names.joblib"):
    assert path.exists(), f"Artifact missing: {path}"
    print(path)